**Project Objective:**
To build an end-to-end Natural Language Processing (NLP) pipeline that automatically sorts raw news articles into distinct topic categories. Using a Recurrent Neural Network (RNN), this project demonstrates how to take messy, real-world text data, translate it into a mathematical format a machine can understand, and train a sequential model to recognize semantic patterns and predict text classifications accurately.

In [1]:
!pip install tensorflow datasets numpy

In [6]:
import numpy as np
import re
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical

#Step 1: Install and Load the Dataset ---
print("Loading dataset...")
dataset = load_dataset('parquet', data_files={'train': '/content/train-00000-of-00001.parquet', 'test': '/content/test-00000-of-00001.parquet'})

# Taking a smaller 5,000-article subset for faster demonstration
train_texts = dataset['train']['text'][:5000]
train_labels = dataset['train']['label'][:5000]
test_texts = dataset['test']['text'][:1000]
test_labels = dataset['test']['label'][:1000]

#Step 2: Clean and Normalize the Text ---
def clean_text(text):
    text = text.lower()
    # Keep only lowercase letters and spaces; strip numbers/punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    return text

print("Cleaning text...")
train_texts_cleaned = [clean_text(text) for text in train_texts]
test_texts_cleaned = [clean_text(text) for text in test_texts]

#Step 3: Tokenize the Strings into Numbers ---
vocab_size = 10000 # Only track the 10,000 most common words

# Create the dictionary lookup
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts_cleaned)

print("Tokenizing...")
train_sequences = tokenizer.texts_to_sequences(train_texts_cleaned)
test_sequences = tokenizer.texts_to_sequences(test_texts_cleaned)

#Step 4: Pad and Truncate Sequences ---
max_length = 50 # Cap all articles at 50 words

print("Padding sequences...")
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

#Step 5: Convert Labels to Categorical Vectors ---
num_classes = 4 # AG News has 4 categories: World, Sports, Business, Sci/Tech

train_labels_cat = to_categorical(train_labels, num_classes=num_classes)
test_labels_cat = to_categorical(test_labels, num_classes=num_classes)

#Step 6: Define the RNN Model Architecture ---
embedding_dim = 64

model = Sequential([
    # Translates word IDs into 64-dimensional semantic vectors
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),

    # Tracks sequential memory across the 50-word phrase
    SimpleRNN(units=32),

    # Outputs the final classification distribution
    Dense(num_classes, activation='softmax')
])

#7:Compile and Train the Model ---
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Training model...")
history = model.fit(
    train_padded,
    train_labels_cat,
    epochs=5,
    validation_split=0.2, # Holds out 20% of the training data to monitor overfitting
    verbose=1
)

#Step 8:Read and Evaluate the Target Accuracy ---
print("\nEvaluating on unseen test data...")
test_loss, test_accuracy = model.evaluate(test_padded, test_labels_cat, verbose=2)
print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Cleaning text...
Tokenizing...
Padding sequences...
Training model...
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.4498 - loss: 1.2130 - val_accuracy: 0.5960 - val_loss: 1.0163
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.7408 - loss: 0.7172 - val_accuracy: 0.6430 - val_loss: 0.9791
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8665 - loss: 0.4306 - val_accuracy: 0.6400 - val_loss: 1.1239
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9247 - loss: 0.2770 - val_accuracy: 0.6650 - val_loss: 1.1758
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9535 - loss: 0.1834 - val_accuracy: 0.6550 - val_loss: 1.3199

Evaluating on unseen test data...
32/32 - 0s - 6ms/step - accuracy: 0.6490 - loss: 1.3179

Final Test Accuracy: 64.90%


**Key Technical Goals Achieved in the Code:**

(1) **Data Normalization:** Stripping out noise (punctuation and capitalization) so the model focuses purely on the vocabulary.

(2) **Vectorization:** Converting human language into dense number arrays via tokenization and word embeddings.

(3) **Sequential Learning:** Using an RNN layer to retain the "memory" of earlier words in a sentence to better understand the overall meaning of the text.

(4) **Performance Metrics:** Creating a strict training loop with a validation split to ensure the model actually learns the underlying patterns rather than just memorizing the training data.